Magnitude Pruning Step-2: Fine-Tuning with 4-bit Llama and QLoRA

In [1]:
import ast
import random
import textwrap
import time
import pandas as pd
import torch
from huggingface_hub import list_repo_files, hf_hub_download
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

base_dir = "/content/pruned_base"
adapter_dir = "/content/pruned_adapter"
dataset_repo = "code-search-net/code_search_net"

n_examples = 400        # "minimal" fine-tune: one pass over 400 examples
max_len = 512           # tokens per example (prompt + answer)
epochs = 1
lr = 2e-4
grad_accum = 8          # effective batch size 8
seed = 0

random.seed(seed)
torch.manual_seed(seed)

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU found. Set the Colab runtime to T4.")

# --- 1. Data: Python functions from CodeSearchNet (validation + test splits only, ~45k functions)
#        that already carry a Google-style docstring (has both "Args:" and "Returns:").
files = [f for f in list_repo_files(dataset_repo, repo_type="dataset", revision="refs/convert/parquet")
         if f.startswith("python/") and f.endswith(".parquet") and ("valid" in f or "test" in f)]
paths = [hf_hub_download(dataset_repo, f, repo_type="dataset", revision="refs/convert/parquet")
         for f in files]
df = pd.concat(pd.read_parquet(p, columns=["whole_func_string", "func_documentation_string"])
               for p in paths)
df = df[df["func_documentation_string"].str.contains("Args:", regex=False)
        & df["func_documentation_string"].str.contains("Returns:", regex=False)
        & ~df["whole_func_string"].str.contains("below_zero", regex=False)]
print(f"Google-style candidates: {len(df):,} functions from {len(files)} parquet files")

def strip_docstring(source):
    """Return (code without its docstring, full code) or None if the function can't be parsed."""
    src = textwrap.dedent(source)
    try:
        fn = ast.parse(src).body[0]
    except (SyntaxError, IndexError):
        return None
    if not isinstance(fn, (ast.FunctionDef, ast.AsyncFunctionDef)) or len(fn.body) < 2:
        return None
    doc = fn.body[0]
    if not (isinstance(doc, ast.Expr) and isinstance(doc.value, ast.Constant)
            and isinstance(doc.value.value, str)):
        return None
    lines = src.splitlines()
    stripped = "\n".join(lines[:doc.lineno - 1] + lines[doc.end_lineno:])
    return stripped, src

prompt_template = ("Generate Google-style docstring for the following Python function: {code}. "
                   "Only output the docstring and function code.")   # P-2 wording, generic functions

tokenizer = AutoTokenizer.from_pretrained("codellama/CodeLlama-7b-Instruct-hf")   # pruning never changes the tokenizer, more stable to load directly from model

examples = []
for source in df["whole_func_string"].sample(frac=1, random_state=seed):
    pair = strip_docstring(source)
    if pair is None:
        continue
    code_in, code_out = pair
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt_template.format(code=code_in)}],
        add_generation_prompt=True, tokenize=False)
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(f" ```\n{code_out}\n```", add_special_tokens=False)["input_ids"] \
        + [tokenizer.eos_token_id]
    if len(prompt_ids) + len(answer_ids) > max_len:
        continue
    # Loss only on the answer: prompt tokens are masked with -100
    examples.append((prompt_ids + answer_ids, [-100] * len(prompt_ids) + answer_ids))
    if len(examples) == n_examples:
        break

print(f"Training examples: {len(examples)} (target {n_examples}, max {max_len} tokens each)")

# --- 2. Model: pruned FP16 weights loaded in 4-bit NF4. NF4 stores 0 exactly, and the base
#        weights stay frozen, so the pruned positions remain zero throughout training.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(base_dir, quantization_config=bnb_config,
                                             device_map="cuda")
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# --- 3. Training loop
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
model.train()
start = time.perf_counter()
step, running = 0, 0.0
for epoch in range(epochs):
    random.shuffle(examples)
    for i, (ids, labels) in enumerate(examples, 1):
        input_ids = torch.tensor([ids], device="cuda")
        label_ids = torch.tensor([labels], device="cuda")
        loss = model(input_ids=input_ids, labels=label_ids).loss
        (loss / grad_accum).backward()
        running += loss.item()
        if i % grad_accum == 0 or i == len(examples):
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            optimizer.step()
            optimizer.zero_grad()
            step += 1
            if step % 5 == 0:
                print(f"epoch {epoch + 1} | step {step} | loss {running / grad_accum:.4f}")
            running = 0.0

train_minutes = (time.perf_counter() - start) / 60
print(f"Training time: {train_minutes:.1f} min ({step} optimizer steps)")

# --- 4. Save the adapter only (tens of MB); step 3 merges it into the FP16 pruned weights
model.save_pretrained(adapter_dir)
print(f"LoRA adapter saved to {adapter_dir}")


Google-style candidates: 2,480 functions from 2 parquet files


<unknown>:10: SyntaxWarning: invalid escape sequence '\d'


Training examples: 400 (target 400, max 512 tokens each)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

trainable params: 19,988,480 || all params: 6,758,535,168 || trainable%: 0.2958


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


epoch 1 | step 5 | loss 0.5541
epoch 1 | step 10 | loss 0.4723
epoch 1 | step 15 | loss 0.5607
epoch 1 | step 20 | loss 0.4847
epoch 1 | step 25 | loss 0.6128
epoch 1 | step 30 | loss 0.7155
epoch 1 | step 35 | loss 0.4508
epoch 1 | step 40 | loss 0.4665
epoch 1 | step 45 | loss 0.3968
epoch 1 | step 50 | loss 0.5253
Training time: 12.3 min (50 optimizer steps)
LoRA adapter saved to /content/pruned_adapter


Download QLoRA

In [3]:
from google.colab import files
files.download("/content/pruned_adapter/adapter_model.safetensors")
files.download("/content/pruned_adapter/adapter_config.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>